In [ ]:
%load_ext autoreload
%autoreload 2

# Swiss roll — checkpoint plotting

Load trained EBiEOT models from Hydra run directories (`checkpoint.pt` + `.hydra/config.yaml`) or from
notebook/CLI checkpoints that embed resolved config in the pickle. Rebuild with `build_neural_model` or
`build_gmm_model`, then call `plot_swiss_roll`.

Run with cwd `EBiEOT/` or `notebooks/swiss_roll/`.

In [ ]:
from src.utils.notebook_setup import ensure_repo_imports, load_train_builders

REPO_ROOT = ensure_repo_imports()
build_gmm_model, build_neural_model = load_train_builders(REPO_ROOT)
import random
import sys
from pathlib import Path
from typing import Any

import numpy as np
import torch
from omegaconf import DictConfig, OmegaConf

from src.utils.plotting.distributions import plot_swiss_roll
from src.utils.samplers.discrete_ot import OTPlanSampler
from src.utils.samplers.synthetic import (
    StandardNormalSampler,
    SwissRollSampler,
    build_swiss_roll_samplers,
)
from src.utils.core.seed import set_seed

@torch.no_grad()
def get_gt_points(
    y_sampler: SwissRollSampler,
    otp_sampler: OTPlanSampler,
    starting_points: torch.Tensor,
    num_ending_points: int = 64,
    pool_size: int = 1024,
) -> list[torch.Tensor]:
    device = starting_points.device
    out: list[torch.Tensor] = []
    for point in starting_points:
        y_pool = y_sampler.sample(pool_size).to(device)
        x_rep = point.unsqueeze(0).expand(pool_size, -1)
        _, y_plan = otp_sampler.sample_plan(x_rep.cpu(), y_pool.cpu())
        out.append(y_plan.to(device)[:num_ending_points])
    return out

def _resolve_cfg(
    run_dir: Path,
    ckpt: dict[str, Any],
    config_path: Path | None = None,
) -> DictConfig:
    if config_path is not None:
        config_path = Path(config_path).expanduser().resolve()
        if not config_path.is_file():
            raise FileNotFoundError(f"config_path not found: {config_path}")
        return OmegaConf.load(config_path)
    hydra_cfg = run_dir / ".hydra" / "config.yaml"
    if hydra_cfg.is_file():
        return OmegaConf.load(hydra_cfg)
    if isinstance(ckpt, dict) and "cfg" in ckpt:
        return OmegaConf.create(ckpt["cfg"])
    raise FileNotFoundError(
        f"No config at {hydra_cfg}, config_path, or checkpoint['cfg'] for {run_dir}"
    )

def _build_model(cfg: DictConfig, device: torch.device) -> torch.nn.Module:
    method = str(cfg.get("method", "neural"))
    if method == "neural":
        return build_neural_model(cfg, device)
    if method == "gmm":
        return build_gmm_model(cfg, device)
    raise ValueError(f"Unknown method={method!r} in config")

def load_model_from_run(
    run_dir: Path,
    device: torch.device,
    *,
    checkpoint_name: str = "checkpoint.pt",
    config_path: Path | None = None,
) -> tuple[torch.nn.Module, DictConfig]:
    run_dir = Path(run_dir).expanduser().resolve()
    ckpt_path = run_dir / checkpoint_name
    if not ckpt_path.is_file():
        raise FileNotFoundError(f"Missing checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = _resolve_cfg(run_dir, ckpt, config_path=config_path)
    model = _build_model(cfg, device)
    state = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
    model.load_state_dict(state)
    model.eval()
    return model, cfg

def run_label(cfg: DictConfig, run_dir: Path, label: str | None) -> str:
    if label:
        return label
    ds = cfg.dataset
    return (
        f"P={ds.P_XY_paired}, Q={ds.Q_X_unpaired}, R={ds.R_Y_unpaired} "
        f"({run_dir.name})"
    )


In [ ]:
device = torch.device(
    f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu"
)
device

In [ ]:
torch.set_default_device(device)
dtype = torch.float64
torch.set_default_dtype(dtype)

## 2. Parameters

Set `RUNS` to Hydra output folders (each must contain `checkpoint.pt`). Optional per-run `label` and
`checkpoint` filename (e.g. notebook training saves `D_<step>.pt` with a sibling `.hydra/config.yaml` if
you copied a Hydra run, or rely on `cfg` inside `checkpoint.pt` from `scripts/train.py`).

Example after `python scripts/train.py experiment=gmm_swiss_roll`:

```python
RUNS = [{"run_dir": REPO_ROOT / "outputs" / "2025-05-15" / "12-00-00"}]
```

In [ ]:
RUNS: list[dict[str, Any]] = [
    # Hydra CLI run (checkpoint.pt + .hydra/config.yaml):
    # {"run_dir": REPO_ROOT / "outputs" / "YYYY-MM-DD" / "HH-MM-SS"},
    # Notebook checkpoint under checkpoints/ (state_dict only; set config_path or use cfg in ckpt):
    # {
    #     "run_dir": REPO_ROOT / "checkpoints" / "<EXP_NAME>",
    #     "checkpoint": "D_3000.pt",
    #     "config_path": REPO_ROOT / "outputs" / "..." / ".hydra" / "config.yaml",
    # },
]

SAVE_DIR = REPO_ROOT / "plots" / "swiss_roll"
NUM_ENDING_POINTS = 64
starting_points = torch.tensor(
    [[-2.0, 0.0], [2.0, 2.0], [0.0, 0.0]],
    device=device,
)

## 3. Load checkpoints

In [ ]:
if not RUNS:
    raise ValueError(
        "Set RUNS to at least one dict with 'run_dir' (Hydra output or checkpoints folder)."
    )

models_dict: dict[str, torch.nn.Module] = {}
loaded_cfgs: list[DictConfig] = []

for spec in RUNS:
    run_dir = Path(spec["run_dir"])
    ckpt_name = str(spec.get("checkpoint", "checkpoint.pt"))
    config_path = spec.get("config_path")
    model, cfg = load_model_from_run(
        run_dir,
        device,
        checkpoint_name=ckpt_name,
        config_path=Path(config_path) if config_path is not None else None,
    )
    title = run_label(cfg, run_dir, spec.get("label"))
    models_dict[title] = model
    loaded_cfgs.append(cfg)
    print(f"Loaded {title} from {run_dir / ckpt_name}")

plot_cfg = loaded_cfgs[0]
seed = int(plot_cfg.seed) if plot_cfg.get("seed") is not None else int(plot_cfg.train.seed)
set_seed(seed)

## 4. Data for plotting

Samplers and paired batches are taken from the first loaded run's dataset config. All runs should use the
same Swiss-roll dataset sizes when comparing multiple checkpoints.

In [ ]:
_, _, pd_sampler = build_swiss_roll_samplers(plot_cfg, device)
X_paired_train, Y_paired_train = pd_sampler.x, pd_sampler.y

ds = plot_cfg.dataset
x_sampler = StandardNormalSampler(dim=int(ds.x_dim), device=str(device))
y_sampler = SwissRollSampler(dim=int(ds.y_dim), device=str(device))
mb = ds.minibatch
otp_sampler = OTPlanSampler(
    method=str(mb.method),
    reg=float(mb.reg),
    reg_m=float(mb.get("reg_m", 1.0)),
    cost_function=str(mb.cost_function),
    normalize_cost=bool(mb.get("normalize_cost", False)),
)

gt_Y_points = get_gt_points(
    y_sampler,
    otp_sampler,
    starting_points,
    num_ending_points=NUM_ENDING_POINTS,
)

## 5. Plot

In [ ]:
plot_swiss_roll(
    models_dict,
    x_sampler,
    y_sampler,
    X_paired_train,
    Y_paired_train,
    starting_points,
    gt_Y_points,
    num_ending_points=NUM_ENDING_POINTS,
    save_dir=str(SAVE_DIR),
)